In [1]:
from tree import Mbtree
from marubatsu import Marubatsu

def calc_playout_prob_by_df(self, node):
    node.playout_prob = {
        node.mb.CIRCLE: 0.0,
        node.mb.CROSS: 0.0,
        node.mb.DRAW: 0.0
    }
    if node.mb.status != Marubatsu.PLAYING:
        node.playout_prob[node.mb.status] = 1.0
        node.playout_score = (node.mb.status == node.mb.CIRCLE) * 1 + \
                             (node.mb.status == node.mb.DRAW) * 0.5
    else:
        childnum = len(node.children)
        for childnode in node.children:
            self.calc_playout_prob_by_df(childnode)
            for status, prob in childnode.playout_prob.items():
                node.playout_prob[status] += prob / childnum  
            node.playout_score = node.playout_prob[node.mb.CIRCLE] + \
                                 node.playout_prob[node.mb.DRAW] * 0.5
            
Mbtree.calc_playout_prob_by_df = calc_playout_prob_by_df

In [2]:
mbtree = Mbtree()
print(mbtree.root.playout_score)

     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946
0.6484126984126983


In [3]:
from tqdm import tqdm
import gzip, pickle

def calc_and_save_bestmoves_and_score_by_board(self, path):
    bestmoves_and_score_by_board = {}
    for node in tqdm(self.nodelist):
        txt = node.mb.board_to_str()
        if not txt in bestmoves_and_score_by_board.keys():
            bestmoves_and_score_by_board[txt] = {
                "bestmoves": [node.mb.board.move_to_xy(move) for move in node.bestmoves],
                "score": node.score,
                "playout_prob": { 
                    node.mb.CIRCLE_STR: node.playout_prob[node.mb.CIRCLE],
                    node.mb.CROSS_STR: node.playout_prob[node.mb.CROSS],
                    node.mb.DRAW: node.playout_prob[node.mb.DRAW],
                },
                "playout_score": node.playout_score,
            }

    with gzip.open(path, "wb") as f:
        pickle.dump(bestmoves_and_score_by_board, f)
    
    return bestmoves_and_score_by_board           

Mbtree.calc_and_save_bestmoves_and_score_by_board = calc_and_save_bestmoves_and_score_by_board

In [4]:
mbtree = Mbtree()
mbtree.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board.dat")
mbtree2 = Mbtree(shortest_victory=True)
mbtree2.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board_shortest_victory.dat")
mbtree3 = Mbtree(shortest_victory=True, recalculate_draw_score=True)
mbtree3.calc_and_save_bestmoves_and_score_by_board("../data/bestmoves_and_score_by_board_sv_rd.dat")

     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:02<00:00, 208834.04it/s]


     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:02<00:00, 211370.43it/s]


     9 depth 1 node created
    72 depth 2 node created
   504 depth 3 node created
  3024 depth 4 node created
 15120 depth 5 node created
 54720 depth 6 node created
148176 depth 7 node created
200448 depth 8 node created
127872 depth 9 node created
     0 depth 10 node created
total node num = 549946


100%|██████████| 549946/549946 [00:02<00:00, 211185.48it/s]


{'.........': {'bestmoves': [(0, 0),
   (0, 1),
   (0, 2),
   (1, 0),
   (1, 1),
   (1, 2),
   (2, 0),
   (2, 1),
   (2, 2)],
  'score': -0.5,
  'playout_prob': {'o': 0.5849206349206348,
   'x': 0.2880952380952381,
   'draw': 0.126984126984127},
  'playout_score': 0.6484126984126983},
 'o........': {'bestmoves': [(1, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.6071428571428571,
   'x': 0.26428571428571423,
   'draw': 0.1285714285714286},
  'playout_score': 0.6714285714285714},
 '.o.......': {'bestmoves': [(0, 0), (0, 2), (1, 1), (2, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.5357142857142857,
   'x': 0.3357142857142857,
   'draw': 0.1285714285714286},
  'playout_score': 0.6},
 '..o......': {'bestmoves': [(1, 1)],
  'score': 0.5,
  'playout_prob': {'o': 0.6071428571428571,
   'x': 0.2642857142857143,
   'draw': 0.1285714285714286},
  'playout_score': 0.6714285714285714},
 '...o.....': {'bestmoves': [(0, 0), (1, 1), (1, 2), (2, 0)],
  'score': 0.5,
  'playout_prob': {'o': 0.535714

In [5]:
from tree import Mbtree_GUI
import matplotlib.pyplot as plt
import ipywidgets as widgets

def create_widgets(self):
    self.output = widgets.Output()  
    self.print_helpmessage()
    self.output.layout.display = "none"
    self.left_button = self.create_button("←", 50)
    self.up_button = self.create_button("↑", 50)
    self.right_button = self.create_button("→", 50)
    self.down_button = self.create_button("↓", 50)
    self.data_dropdown= widgets.Dropdown(
        options = {
            "表示なし": False,
            "評価値": "score",
            "プレイアウトでの〇の勝率": self.selectednode.mb.CIRCLE_STR,
            "プレイアウトでの×の勝率": self.selectednode.mb.CROSS_STR,
            "プレイアウトでの引き分け率": self.selectednode.mb.DRAW,
            "プレイアウトの〇の評価値": "playout_score_o",
            "プレイアウトの×の評価値": "playout_score_x",
        },
        description = "データの表示",
        value = "score"
    )
    self.size_slider = widgets.FloatSlider(min=0.05, max=0.25, step=0.01, description="size", value=self.size)
    self.help_button = self.create_button("？", 50)
    self.label = widgets.Label(value="", layout=widgets.Layout(width=f"50px"))
    
    with plt.ioff():
        self.fig = plt.figure(figsize=[self.width * self.size,
                                        self.height * self.size])
        self.ax = self.fig.add_axes([0, 0, 1, 1])
    self.fig.canvas.toolbar_visible = False
    self.fig.canvas.header_visible = False
    self.fig.canvas.footer_visible = False
    self.fig.canvas.resizable = False    
    
    self.dropdown = widgets.Dropdown(
        options=self.scoretable_dict,
        description="score table",
    )
    self.bestmoves_and_score_by_board = self.dropdown.value   
    
Mbtree_GUI.create_widgets = create_widgets

In [6]:
from tree import Node, Rect
from marubatsu import Marubatsu_GUI

def __init__(self, mb, parent=None, depth=0, bestmoves_and_score_by_board=None):
    self.id = Node.count
    Node.count += 1
    self.mb = mb
    self.parent = parent
    self.depth = depth
    self.children = []
    self.children_by_move = {}   
    if bestmoves_and_score_by_board is not None:
        bestmoves_and_score = bestmoves_and_score_by_board[self.mb.board_to_str()]
        self.bestmoves = bestmoves_and_score["bestmoves"]
        self.score = bestmoves_and_score["score"]   
        self.playout_prob = bestmoves_and_score["playout_prob"]   
        self.playout_score = bestmoves_and_score["playout_score"]
        
Node.__init__ = __init__

def draw_node(self, ax=None, maxdepth=None, emphasize=False, darkness=0, 
              show_score=True, size=0.25, lw=0.8, dx=0, dy=0):
    width = 8
    if ax is None:
        height = len(self.children) * 4
        fig, ax = plt.subplots(figsize=(width * size, height * size))
        ax.set_xlim(0, width)
        ax.set_ylim(0, height)   
        ax.invert_yaxis()
        ax.axis("off")
        for childnode in self.children:
            childnode.height = 4
        self.height = height         
        
    # 自分自身のノードを真ん中の位置になるように (dx, dy) からずらして描画する
    y = dy + (self.height - 3) / 2
    bc = "red" if emphasize else None
    Marubatsu_GUI.draw_board(ax, self.mb, show_result=True, 
                            score=getattr(self, "score", None), bc=bc, darkness=darkness, lw=lw, dx=dx, dy=y)
    if hasattr(self, "playout_prob"):
        if show_score in [self.mb.CIRCLE_STR, self.mb.CROSS_STR, self.mb.DRAW]:
            ax.text(dx, y - 0.1, f"{self.playout_prob[show_score]:.3f}", fontsize=70*size)
    if hasattr(self, "playout_score"):
        if show_score == "playout_score_o":
            ax.text(dx, y - 0.1, f"{self.playout_score:.3f}", fontsize=70*size)
        if show_score == "playout_score_x":
            ax.text(dx, y - 0.1, f"{1 - self.playout_score:.3f}", fontsize=70*size)
    if hasattr(self, "score") and show_score in [True, "score"]:
        ax.text(dx, y - 0.1, self.score, fontsize=70*size)
    rect = Rect(dx, y, 3, 3)
    # 子ノードが存在する場合に、エッジの線と子ノードを描画する
    if len(self.children) > 0:
        if maxdepth != self.depth:   
            ax.plot([dx + 3.5, dx + 4], [y + 1.5, y + 1.5], c="k", lw=lw)
            prevy = None
            for childnode in self.children:
                childnodey = dy + (childnode.height - 3) / 2
                if maxdepth is None:
                    Marubatsu_GUI.draw_board(ax, childnode.mb, show_result=True,
                                            score=getattr(childnode, "score", None), dx=dx+5, dy=childnodey, lw=lw)
                edgey = childnodey + 1.5
                ax.plot([dx + 4 , dx + 4.5], [edgey, edgey], c="k", lw=lw)
                if prevy is not None:
                    ax.plot([dx + 4 , dx + 4], [prevy, edgey], c="k", lw=lw)
                prevy = edgey
                dy += childnode.height
        else:
            ax.plot([dx + 3.5, dx + 4.5], [y + 1.5, y + 1.5], c="k", lw=lw)
            
    return rect

Node.draw_node = draw_node

In [7]:
Mbtree_GUI()

In [ ]:
from ai import dprint
from random import choice

def ai_pmc2(mb, pnum, timelimit=None, debug=False, analyze=False, *args, **kwargs):
    if mb.move_count == 8:
        best_move = mb.calc_legal_moves()[0]
        if analyze:
            return {
                "candidate": [mb.board.move_to_xy(best_move)],
                "ratio_by_move": {},
                "playout num": 0,
                "score_by_move": {best_move: 1},                
            }
        else:
            return best_move
    retval = mb.playout(pnum, timelimit)
    best_moves = []
    best_movesxy = []
    best_score = -1
    if analyze:
        score_by_move = {}
    for move, count in retval["result"].items():
        totalcount = max(1, sum(count.values()))
        score = (count[mb.turn] + count[mb.DRAW] * 0.5) / totalcount
        movexy = mb.board.move_to_xy(move)
        dprint(debug, "=" * 50)
        dprint(debug, f"move {movexy}")
        dprint(debug, f"score     : {score:.3f}")
        dprint(debug, f"best score: {best_score:.3f}", )
        if score > best_score:
            best_score = score
            best_moves = [move]
            best_movesxy = [movexy]
            dprint(debug, "UPDATE")
            dprint(debug, f"  best score {best_score}")
            dprint(debug, f"  best moves {best_movesxy}")
        elif score == best_score:
            best_moves.append(move)
            best_movesxy.append(movexy)
            dprint(debug, "APPEND")
            dprint(debug, f"  best moves {best_movesxy}")
        if analyze:
            score_by_move[move] = round(score, 3)
    if analyze:
        return {
            "candidate": best_movesxy,
            "playout num": retval["count"],
            "score_by_move": score_by_move
        }
    else:
        return choice(best_moves)  

In [9]:
mb = Marubatsu()
ai_pmc2(mb, pnum=10000, debug=True, analyze=True)

move (0, 0)
score     : 0.665
best score: -1.000
UPDATE
  best score 0.6653577661431065
  best moves [(0, 0)]
move (0, 1)
score     : 0.612
best score: 0.665
move (0, 2)
score     : 0.661
best score: 0.665
move (1, 0)
score     : 0.613
best score: 0.665
move (1, 1)
score     : 0.754
best score: 0.665
UPDATE
  best score 0.7541743970315399
  best moves [(1, 1)]
move (1, 2)
score     : 0.607
best score: 0.754
move (2, 0)
score     : 0.696
best score: 0.754
move (2, 1)
score     : 0.603
best score: 0.754
move (2, 2)
score     : 0.653
best score: 0.754


{'candidate': [(1, 1)],
 'playout num': 10000,
 'score_by_move': {(0, 0): 0.665,
  (0, 1): 0.612,
  (0, 2): 0.661,
  (1, 0): 0.613,
  (1, 1): 0.754,
  (1, 2): 0.607,
  (2, 0): 0.696,
  (2, 1): 0.603,
  (2, 2): 0.653}}

In [10]:
from ai import ai_match, ai2s, ai14s

for pnum in [5, 10, 50, 100, 1000, 10000]:
    print(f"pnum {pnum}")
    ai_match(ai=[ai_pmc2, ai2s], params=[{"pnum": pnum}, {}], match_num=1000)
    
for timelimit in [0.0001, 0.0005, 0.001, 0.01, 0.1]:
    print(f"timelimit {timelimit}")
    ai_match(ai=[ai_pmc2, ai2s], params=[{"pnum": 100000000, "timelimit": timelimit}, {}],
             match_num=1000)

for pnum in [5, 10, 50, 100, 1000, 10000]:
    print(f"pnum {pnum}")
    ai_match(ai=[ai_pmc2, ai14s], params=[{"pnum": pnum}, {}], match_num=1000)
    
for timelimit in [0.0001, 0.0005, 0.001, 0.01, 0.1]:
    print(f"timelimit {timelimit}")
    ai_match(ai=[ai_pmc2, ai14s], params=[{"pnum": 100000000, "timelimit": timelimit}, {}],
             match_num=1000)

pnum 5
ai_pmc2 VS ai2s


100%|██████████| 1000/1000 [00:00<00:00, 1673.41it/s]


count     win    lose    draw
o         781     129      90
x         576     292     132
total    1357     421     222

ratio     win    lose    draw
o       78.1%   12.9%    9.0%
x       57.6%   29.2%   13.2%
total   67.8%   21.1%   11.1%

pnum 10
ai_pmc2 VS ai2s


100%|██████████| 1000/1000 [00:00<00:00, 1172.12it/s]


count     win    lose    draw
o         850      87      63
x         684     223      93
total    1534     310     156

ratio     win    lose    draw
o       85.0%    8.7%    6.3%
x       68.4%   22.3%    9.3%
total   76.7%   15.5%    7.8%

pnum 50
ai_pmc2 VS ai2s


...

count     win    lose    draw
o           0      81     919
x           0     664     336
total       0     745    1255

ratio     win    lose    draw
o        0.0%    8.1%   91.9%
x        0.0%   66.4%   33.6%
total    0.0%   37.2%   62.7%

timelimit 0.01
ai_pmc2 VS ai14s


100%|██████████| 1000/1000 [01:16<00:00, 13.05it/s]


count     win    lose    draw
o           0       0    1000
x           0     474     526
total       0     474    1526

ratio     win    lose    draw
o        0.0%    0.0%  100.0%
x        0.0%   47.4%   52.6%
total    0.0%   23.7%   76.3%

timelimit 0.1
ai_pmc2 VS ai14s


100%|██████████| 1000/1000 [12:30<00:00,  1.33it/s]

count     win    lose    draw
o           0       0    1000
x           0     516     484
total       0     516    1484

ratio     win    lose    draw
o        0.0%    0.0%  100.0%
x        0.0%   51.6%   48.4%
total    0.0%   25.8%   74.2%



In [11]:
checkedmb = set()
nodelist = []
for node in mbtree.nodelist:
    if node.mb.board_to_hashable() in checkedmb:
        continue
    checkedmb |= node.mb.board.calc_same_hashables()
    bestmoves = []
    bestscore = -1
    for move in node.mb.calc_legal_moves():
        score = node.children_by_move[move].playout_score
        if node.mb.turn == mb.CROSS:
            score = 1 - score
        if score > bestscore:
            bestmoves = [move]
            bestscore = score
        elif score == bestscore:
            bestmoves.append(move)
    if not(set(bestmoves) <= set(node.bestmoves)):
        nodelist.append((node, set(bestmoves), set(node.bestmoves)))

print(len(nodelist), len(mbtree.nodelist))

26 549946


In [12]:
for node, pmcmoves, bestmoves in nodelist:
    pmcmoves = [node.mb.board.move_to_xy(move) for move in pmcmoves]
    bestmoves = [node.mb.board.move_to_xy(move) for move in bestmoves]
    print(f"pmcmoves  {pmcmoves}")
    print(f"bestmoves {bestmoves}")
    print(node.mb)

pmcmoves  [(1, 1)]
bestmoves [(1, 0), (2, 2), (2, 0)]
Turn o
o..
...
X..

pmcmoves  [(1, 1)]
bestmoves [(0, 0)]
Turn o
..X
o..
...

pmcmoves  [(1, 1), (2, 2)]
bestmoves [(2, 0), (2, 2)]
Turn x
o..
x.O
...

pmcmoves  [(1, 1)]
bestmoves [(1, 2), (2, 2)]
Turn x
o..
O..
x..

pmcmoves  [(1, 1)]
bestmoves [(2, 2)]
Turn x
o..
..O
x..

pmcmoves  [(2, 0)]
bestmoves [(2, 1), (2, 2), (0, 1), (0, 2)]
Turn x
o..
.x.
.O.

pmcmoves  [(2, 0), (0, 2)]
bestmoves [(1, 0), (0, 1), (1, 2), (2, 1)]
Turn x
o..
.x.
..O

pmcmoves  [(2, 0)]
bestmoves [(2, 1)]
Turn x
x..
oO.
...

pmcmoves  [(1, 1)]
bestmoves [(2, 0)]
Turn x
x..
o..
.O.

pmcmoves  [(2, 0)]
bestmoves [(2, 2), (0, 2)]
Turn x
.x.
o..
.O.

pmcmoves  [(1, 1)]
bestmoves [(2, 2), (0, 0)]
Turn x
..x
o..
.O.

pmcmoves  [(1, 0), (2, 0)]
bestmoves [(1, 0)]
Turn o
o..
...
xoX

pmcmoves  [(1, 1), (2, 2)]
bestmoves [(2, 2)]
Turn x
ox.
x..
oO.

pmcmoves  [(1, 1), (2, 2), (2, 1)]
bestmoves [(2, 1)]
Turn x
oOx
x..
o..

pmcmoves  [(2, 0), (2, 2)]
bestmoves [(2, 0)

In [13]:
import ai as ai_module
from ai import ai_gt7, ai_pmc
from util import load_bestmoves

def gui_play(ai=None, params=None, ai_dict=None, mbparams={}, seed=None):
    # ai が None の場合は、人間どうしの対戦を行う
    if ai is None:
        ai = [None, None]
    if params is None:
        params = [{}, {}]
    # ai_dict が None の場合は、ai1s ~ ai14s の Dropdown を作成するためのデータを計算する
    if ai_dict is None:
        ai_dict = { "人間": ( None, {} ) }
        for i in range(1, 15):
            ai_name = f"ai{i}s"  
            ai_dict[ai_name] = (getattr(ai_module, ai_name), {})
        bestmoves_and_score_by_board = load_bestmoves("../data/bestmoves_and_score_by_board.dat")
        ai_dict["ai_gt7"] = (ai_gt7, {"bestmoves_and_score_by_board": bestmoves_and_score_by_board})
        bestmoves_and_score_by_board_sv = load_bestmoves("../data/bestmoves_and_score_by_board_shortest_victory.dat")
        ai_dict["ai_gtsv"] = (ai_gt7, {"bestmoves_and_score_by_board": bestmoves_and_score_by_board_sv})
        bestmoves_and_score_by_board_svrd = load_bestmoves("../data/bestmoves_and_score_by_board_sv_rd.dat")
        ai_dict["ai_gtsvrd"] = (ai_gt7, {"bestmoves_and_score_by_board": bestmoves_and_score_by_board_svrd})
        for pnum in [5, 10, 100, 1000, 10000]:
            ai_dict[f"ai_pmc({pnum})"] = (ai_pmc, {"pnum": pnum})
        for pnum in [5, 10, 100, 1000, 10000]:
            ai_dict[f"ai_pmc2({pnum})"] = (ai_pmc2, {"pnum": pnum})

    mb = Marubatsu(**mbparams)
    mb.play(ai=ai, params=params, ai_dict=ai_dict, seed=seed, gui=True)

In [14]:
gui_play()